In [73]:
import scanpy as sc
import scvi
import numpy as np
import sys

sys.path.append('../')

from scripts.subset_hvg import subset_to_hvg

adata = sc.read_h5ad(
    "../../data/obesity_subset_CEBPB_KIF11.h5ad"
)

print(adata)

adata_subset, hvg_genes, sig_genes = subset_to_hvg(
    adata,
    hvg_path= "../../data/preprocessed/HVG/hvg5000_genes.txt",
    include_signature_genes=True
)


print(adata_subset)

AnnData object with n_obs × n_vars = 24325 × 36601
    obs: 'nCount_RNA', 'nFeature_RNA', 'nCount_guide', 'nFeature_guide', 'percent.mt', 'SampleID', 'Day', 'num_features', 'feature_call', 'num_umis', 'gene', 'adipo', 'pre_adipo', 'other', 'lipo'
    uns: 'log1p'
    layers: 'counts'
HVG requested: 5000
Signature genes requested: 820
Signature genes found in dataset: 797
Total genes used: 5563
Missing genes: 0
AnnData object with n_obs × n_vars = 24325 × 5563
    obs: 'nCount_RNA', 'nFeature_RNA', 'nCount_guide', 'nFeature_guide', 'percent.mt', 'SampleID', 'Day', 'num_features', 'feature_call', 'num_umis', 'gene', 'adipo', 'pre_adipo', 'other', 'lipo'
    uns: 'log1p'
    layers: 'counts'


In [74]:
import numpy as np

def create_scanvi_labels(adata):

    labels = []

    for _, row in adata.obs.iterrows():

        if row["lipo"] == 1:
            labels.append("lipo")

        elif row["adipo"] == 1:
            labels.append("adipo")

        elif row["pre_adipo"] == 1:
            labels.append("pre_adipo")

        else:
            labels.append("other")

    adata.obs["cell_state"] = labels

    return adata


adata_subset = create_scanvi_labels(adata_subset)

print(adata_subset.obs["cell_state"].value_counts())

cell_state
other        12084
pre_adipo     7379
adipo         3824
lipo          1038
Name: count, dtype: int64


In [75]:
train_perts = [
    "NC",
    "NC+NC",
    "CEBPB",
    "KIF11",
    "CEBPB+NC",
    "KIF11+NC"
]

adata_train = adata_subset[
    adata_subset.obs["gene"].isin(train_perts)
].copy()

adata_test = adata_subset[
    adata_subset.obs["gene"] == "CEBPB+KIF11"
].copy()

# PCA

In [ ]:
from scripts.pca import compute_pca_latent



adata_train, scaler = compute_pca_latent(adata_train, 512)

print(adata_train.obsm["X_pca"].shape)

save in pca_512.joblib
(24283, 512)


In [ ]:
import numpy as 
from sklearn.decomposition import PCA



X_test = adata_test.X

if not isinstance(X_test, np.ndarray):
    X_test = X_test.toarray()

z_test = scaler.transform(X_test)

adata_test.obsm["X_pca"] = z_test

print(adata_test.obsm["X_pca"].shape)

(42, 512)


# Scvi

In [ ]:
import scvi

# setup anndata
scvi.model.SCVI.setup_anndata(
    adata_train,
    layer="counts"
)

# 先訓練 unsupervised VAE
vae = scvi.model.SCVI(
    adata_train,
    n_latent=50
)

vae.train(
    max_epochs=200,
    accelerator="mps",
    devices=1,
    batch_size=256,
    early_stopping=True
)

# 再轉成 semi-supervised model
scanvi = scvi.model.SCANVI.from_scvi_model(
    vae,
    labels_key="cell_state",
    unlabeled_category="unknown"   # 必須是字串
)

scanvi.train(
    max_epochs=200,
    accelerator="mps",
    devices=1,
    batch_size=256,
    early_stopping=True
)

# 取得 latent embedding
adata_train.obsm["X_scanvi"] = scanvi.get_latent_representation(adata_train)

print(adata_train.obsm["X_scanvi"].shape)

/opt/anaconda3/envs/X-Cells/lib/python3.11/site-packages/scvi/train/_trainrunner.py:98: UserWarning: `accelerator` has been set to `mps`. Please note that not all PyTorch/Jax operations are supported with this backend. as a result, some models might be slower and less accurate than usual. Please verify your analysis!Refer to https://github.com/pytorch/pytorch/issues/77764 for more details.
  accelerator, lightning_devices, device = parse_device_args(
GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/opt/anaconda3/envs/X-Cells/lib/python3.11/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/opt/anaconda3/envs

Epoch 200/200: 100%|██████████| 200/200 [11:12<00:00,  3.27s/it, v_num=1, train_loss=3.79e+3]

`Trainer.fit` stopped: `max_epochs=200` reached.


Epoch 200/200: 100%|██████████| 200/200 [11:12<00:00,  3.36s/it, v_num=1, train_loss=3.79e+3]


ValueError: Categorical categories cannot be null

In [15]:
save_dir = "model/scanvi_model"

scanvi.save(
    save_dir,
    overwrite=True
)

print("model saved to:", save_dir)

model saved to: model/scanvi_model


In [19]:
# load model
scanvi = scvi.model.SCANVI.load(
    "model/scanvi_model",
    adata=adata_train
)

# mapping
scvi.model.SCANVI.prepare_query_anndata(
    adata_test,
    scanvi
)

# latent
adata_test.obsm["X_scanvi"] = (
    scanvi.get_latent_representation(
        adata_test
    )
)

print("latent shape:")
print(adata_test.obsm["X_scanvi"].shape)

INFO     File model/scanvi_model/model.pt already downloaded                                                       
INFO     Found 100.0% reference vars in query data.                                                                
INFO     AnnData object appears to be a copy. Attempting to transfer setup.                                        
latent shape:
(42, 50)


/opt/anaconda3/envs/X-Cells/lib/python3.11/site-packages/scvi/model/base/_base_model.py:862: UserWarning: `accelerator` has been automatically set to `cpu` although 'mps' exists. If you wish to run on mps backend, use explicitly accelerator='mps' in train function.In future releases it will become default for mps supported machines.
  _, _, device = parse_device_args(
/opt/anaconda3/envs/X-Cells/lib/python3.11/site-packages/scvi/data/fields/_dataframe_field.py:227: UserWarning: Category 1 in adata.obs['_scvi_labels'] has fewer than 3 cells. Models may not train properly.
  new_mapping = _make_column_categorical(
/opt/anaconda3/envs/X-Cells/lib/python3.11/site-packages/scvi/data/fields/_scanvi.py:56: UserWarning: Category 1 in adata.obs['_scvi_labels'] has fewer than 3 cells. Models may not train properly.
  mapping = _make_column_categorical(


In [22]:
adata_train.obsm["X_scanvi"], adata_test.obsm["X_scanvi"]

(array([[ 2.4879851 ,  0.7558305 , -0.00999486, ...,  1.3833537 ,
         -0.86709535,  1.5022454 ],
        [-0.22768128, -0.71727633,  0.83540416, ..., -0.36426836,
          1.8049107 ,  0.25207883],
        [-0.37686682,  0.53008616,  0.46941304, ..., -1.730259  ,
         -0.7751216 ,  0.50506145],
        ...,
        [-0.7135123 , -0.40143   , -1.2838624 , ..., -0.9500755 ,
          0.32479012,  0.31852138],
        [ 0.5698017 ,  0.4026513 , -0.6552714 , ..., -0.3661348 ,
          1.3828532 , -0.00605619],
        [-0.27258438,  0.11372659, -0.7446291 , ..., -0.7684933 ,
         -0.32492328,  0.50440043]], shape=(24283, 50), dtype=float32),
 array([[ 0.991272  , -0.78429943, -0.38619363, ..., -0.12610286,
          1.2229611 , -0.36384153],
        [ 0.1886192 , -0.08552489, -0.9740019 , ..., -1.3146505 ,
         -0.9784936 ,  0.27087015],
        [ 1.0839583 ,  0.61385494, -1.5835121 , ..., -0.12586212,
          0.54875946, -0.5530902 ],
        ...,
        [ 0.69317484

# Classifer

In [78]:
from sklearn.model_selection import train_test_split

X = adata_train.obsm["X_pca"]
y = adata_train.obs["cell_state"]

X_train, X_val, y_train, y_val = train_test_split(

    X,
    y,

    test_size=0.2,

    stratify=y,

    random_state=42
)

from lightgbm import LGBMClassifier
from sklearn.metrics import classification_report
from sklearn.metrics import accuracy_score


clf = LGBMClassifier(

    n_estimators=300,

    learning_rate=0.05,

    num_leaves=64,

    random_state=42
)

clf.fit(

    X_train,

    y_train,

    eval_set=[(X_val, y_val)],

    eval_metric="multi_logloss"
)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.015475 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 130560
[LightGBM] [Info] Number of data points in the train set: 19426, number of used features: 512
[LightGBM] [Info] Start training from score -1.850488
[LightGBM] [Info] Start training from score -3.152942
[LightGBM] [Info] Start training from score -0.699758
[LightGBM] [Info] Start training from score -1.192677


,boosting_type,'gbdt'
,num_leaves,64
,max_depth,-1
,learning_rate,0.05
,n_estimators,300
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [79]:
y_pred = clf.predict(X_val)

print("VAL accuracy:")
print(accuracy_score(y_val, y_pred))

print("\nclassification report")
print(classification_report(y_val, y_pred))

VAL accuracy:
0.84414247477867

classification report
              precision    recall  f1-score   support

       adipo       0.83      0.82      0.83       764
        lipo       0.90      0.74      0.81       207
       other       0.83      0.90      0.86      2412
   pre_adipo       0.87      0.78      0.82      1474

    accuracy                           0.84      4857
   macro avg       0.86      0.81      0.83      4857
weighted avg       0.85      0.84      0.84      4857



/opt/anaconda3/envs/X-Cells/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [80]:
test_pred = clf.predict(

    adata_test.obsm["X_pca"]
)

adata_test.obs["pred_cell_state"] = test_pred

print(

    adata_test.obs["pred_cell_state"].value_counts()

)

pred_cell_state
other        23
pre_adipo    10
adipo         8
lipo          1
Name: count, dtype: int64


/opt/anaconda3/envs/X-Cells/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [81]:
from sklearn.metrics import classification_report
from sklearn.metrics import accuracy_score
import pandas as pd


y_true = adata_test.obs["cell_state"]
y_pred = adata_test.obs["pred_cell_state"]

print("Accuracy")
print(accuracy_score(y_true, y_pred))

print("\nClassification report")
print(
    classification_report(
        y_true,
        y_pred,
        digits=4
    )
)

print("\nConfusion matrix (row-normalized)")
print(
    pd.crosstab(
        y_true,
        y_pred,
        normalize="index"
    )
)

Accuracy
0.8095238095238095

Classification report
              precision    recall  f1-score   support

       adipo     0.6250    0.7143    0.6667         7
        lipo     0.0000    0.0000    0.0000         1
       other     0.8696    0.8696    0.8696        23
   pre_adipo     0.9000    0.8182    0.8571        11

    accuracy                         0.8095        42
   macro avg     0.5986    0.6005    0.5983        42
weighted avg     0.8161    0.8095    0.8118        42


Confusion matrix (row-normalized)
pred_cell_state     adipo      lipo     other  pre_adipo
cell_state                                              
adipo            0.714286  0.142857  0.142857   0.000000
lipo             1.000000  0.000000  0.000000   0.000000
other            0.086957  0.000000  0.869565   0.043478
pre_adipo        0.000000  0.000000  0.181818   0.818182


In [33]:
from sklearn.metrics import classification_report
from sklearn.metrics import accuracy_score
import pandas as pd


y_true = adata_test.obs["cell_state"]
y_pred = adata_test.obs["pred_cell_state"]

print("Accuracy")
print(accuracy_score(y_true, y_pred))

print("\nClassification report")
print(
    classification_report(
        y_true,
        y_pred,
        digits=4
    )
)

print("\nConfusion matrix (row-normalized)")
print(
    pd.crosstab(
        y_true,
        y_pred,
        normalize="index"
    )
)

Accuracy
0.8571428571428571

Classification report
              precision    recall  f1-score   support

       adipo     0.7500    0.8571    0.8000         7
        lipo     0.0000    0.0000    0.0000         1
       other     0.8462    0.9565    0.8980        23
   pre_adipo     1.0000    0.7273    0.8421        11

    accuracy                         0.8571        42
   macro avg     0.6490    0.6352    0.6350        42
weighted avg     0.8503    0.8571    0.8456        42


Confusion matrix (row-normalized)
pred_cell_state     adipo     other  pre_adipo
cell_state                                    
adipo            0.857143  0.142857   0.000000
lipo             1.000000  0.000000   0.000000
other            0.043478  0.956522   0.000000
pre_adipo        0.000000  0.272727   0.727273


/opt/anaconda3/envs/X-Cells/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/anaconda3/envs/X-Cells/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/anaconda3/envs/X-Cells/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", res

In [86]:
from scripts.scoring import compute_metric_l1_distance
from scripts.cal_propotion import compute_proportion_df

gtruth_proportion = compute_proportion_df(

    adata_test,

    "cell_state"

)

predicted_proportion = compute_proportion_df(

    adata_test,

    "pred_cell_state"

)
l1_distance = compute_metric_l1_distance(

    gtruth_proportion,

    predicted_proportion,

)

print(f"L1-distance score: {l1_distance:.4f}")

L1-distance score: 0.0402


In [88]:
predicted_proportion 

,gene,pre_adipo,adipo,other,lipo,lipo_adipo
0,CEBPB+KIF11,0.238095,0.190476,0.547619,0.02381,0.125
